# 6 - Predicted ICI Response Landscape

**Purpose**: Load the combined CIDER T-cell Score + TMB logistic regression
model fit in `9_roc.ipynb` (not refit here -- see Section 1), then project
predicted response probabilities across all ~8,000 Hartwig patients to
identify which cancer types carry a biomarker profile most consistent with
ICI responsiveness.

Loading the shared model rather than independently refitting one means
Panel C's predictions come from the exact same model and training
population as Figure 5B (`9_roc.ipynb`) -- same formula
(`response ~ log2(CIDER + 1) + log2(TMB + 1)`). Panel C's title also
reports the same cross-validated "AUC x.xx ± y.yy" (not an in-sample
point estimate) as Figure 5B's "TMB + CIDER Score" label, read directly
from 9_roc.ipynb's saved CV results rather than recomputed here.

| | |
|---|---|
| **Inputs** | `CIDER_PROJ_DIR/cpi_ready.rds`, `ready.rds`, `figs/tmb_cider_combined_model.rds` |
| **Outputs** | `figs/model_schematic_panel.rds`, `figs/predicted_response_cpi_cohorts.rds`, `figs/predicted_response_landscape.rds/png` (+ `_no_overlay.rds/png`), `figs/predicted_vs_orr.rds/png` |
| **Run after** | `4_base.ipynb`, `9_roc.ipynb` (for the shared model) |

In [ ]:
source(paste0(dirname(getwd()), "/helpers/map.r"))
source(paste0(dirname(getwd()), "/helpers/shortcuts.r"))
source(paste0(dirname(getwd()), "/helpers/config.r"))
source(paste0(dirname(getwd()), "/helpers/themes.r"))

library(pROC)
library(patchwork)
library(RColorBrewer)
library(magick)   # for loading SVG schematic
library(ggpubr)   # for stat_compare_means

dir.create(paste0(CIDER_PROJ_DIR, "figs/"), showWarnings = FALSE, recursive = TRUE)

# 0 - Load data

# 1 - Load model schematic (Panel A)

Reads the SVG schematic from `fig_references/schematic_figs/model_schematic.svg`
and wraps it as a ggplot-compatible panel for assembly.

In [ ]:
# Load the model schematic as a ggplot-compatible panel.
# model_schematic.svg is a small (470x230 viewBox) pure-vector SVG, rasterized
# here via magick (no raster-native alternative for annotation_raster()).
# image_read_svg()'s signature is (path, width, height) -- it has NO
# `density` argument (that belongs to the generic image_read(), not this
# SVG-specific one; passing density= throws "unused argument"). width = 6000
# is a ~12.8x upscale of the 470px-wide viewBox (height scales automatically
# to preserve the ~2:1 aspect ratio), well above what's needed even at this
# figure's full 300dpi print size, so Panel A stays sharp regardless of how
# large it ends up in the assembled figure_7.png.
schematic_path <- file.path(dirname(getwd()), "fig_references", "schematic_figs", "model_schematic.svg")
if (!file.exists(schematic_path))
  stop("Missing schematic: fig_references/schematic_figs/model_schematic.svg not found in repo root.")

schematic_img <- magick::image_read_svg(schematic_path, width = 6000)
message("Schematic rasterized at: ",
        paste(magick::image_info(schematic_img)[c("width", "height")], collapse = "x"), "px")

# Wrap as a blank ggplot with the image drawn over it
p_schematic <- ggplot() +
  annotation_raster(
    as.raster(schematic_img),
    xmin = 0, xmax = 1, ymin = 0, ymax = 1
  ) +
  scale_x_continuous(limits = c(0, 1), expand = c(0, 0)) +
  scale_y_continuous(limits = c(0, 1), expand = c(0, 0)) +
  theme_void()

saveRDS(p_schematic, paste0(CIDER_PROJ_DIR, "figs/model_schematic_panel.rds"))
set_plot_dims(10, 4); p_schematic

In [ ]:
cpi   <- readRDS(paste0(CIDER_PROJ_DIR, "cpi_ready.rds"))
ready <- readRDS(paste0(CIDER_PROJ_DIR, "ready.rds"))

# Split cpi's cohort_broad residual "Other" bucket into Non-CRC MSI / Other
# (MSS) -- Panel B below (predicted_response_cpi_cohorts, the cohort panel
# in Figure 7) otherwise shows one "Other" bar mixing MSI-high and
# microsatellite-stable patients from many different tumor types (same
# split as Figure 3/4; see helpers/shortcuts.r::split_other_by_msi()).
# Level order: BROAD_LEVELS_OTHER_SPLIT (helpers/config.r). Display-only
# and local to this notebook's own copy of `cpi` -- ready/predict_df
# (Panel C, which uses cohort_pan instead) are untouched.
cohort_other_split_levels <- c(
  BROAD_LEVELS_OTHER_SPLIT,
  intersect("Other (Unknown MSI)", unique(split_other_by_msi(cpi$cohort_broad, cpi$msi_high)))
)
cpi <- cpi |>
  mutate(cohort_broad = factor(
    split_other_by_msi(cohort_broad, msi_high),
    levels = cohort_other_split_levels
  )) |>
  group_by(cohort_broad) |>
  mutate(n_broad = n_distinct(sampleId)) |>
  ungroup() |>
  mutate(
    cohort_broad_go = {
      lkp <- distinct(pick(cohort_broad, n_broad)) |> arrange(cohort_broad)
      factor(
        paste0(cohort_broad, " (N = ", n_broad, ")"),
        levels = paste0(lkp$cohort_broad, " (N = ", lkp$n_broad, ")")
      )
    }
  ) |>
  select(-n_broad)

# 1 - Load the combined model (fit in 9_roc.ipynb)

Model: `response ~ cider_log2 + tmb_log2`, i.e. `log2(cider_score_depth_adj + 1)
+ log2(tmbPerMb + 1)` -- log2 for both, so a one-unit change in either
predictor means the raw value doubled. Fit once, in `9_roc.ipynb`, on
pan-cancer ICI-treated patients with T-cell scores and known response
(one row per patient -- the "ICI-Treated" aggregate duplicate is excluded
there); loaded here rather than refit, so this notebook's predictions come
from the identical model. The AUC shown in Panel C's title is also read
from 9_roc.ipynb (the cross-validated mean ± SD), matching Figure 5B's
"TMB + CIDER Score" label exactly.

In [ ]:
# The combined TMB + CIDER Score model is fit once, in 9_roc.ipynb, and
# loaded here rather than refit -- same model object, same training
# population, same formula (response ~ log2(CIDER + 1) + log2(TMB + 1)) --
# so the "TMB + CIDER Score" AUC reported here (Figure 7 Panel C) and in
# Figure 5B can never drift apart. See 9_roc.ipynb's "Overall (pan-cancer)
# ROC" section for how the model and its training population (one row per
# ICI-treated patient, T-cell group, response known) are built.
tmb_cider_file <- paste0(CIDER_PROJ_DIR, "figs/tmb_cider_combined_model.rds")
if (!file.exists(tmb_cider_file))
  stop("Missing figs/tmb_cider_combined_model.rds -- run 9_roc.ipynb before this notebook.")

tmb_cider <- readRDS(tmb_cider_file)
model     <- tmb_cider$model
train_df  <- tmb_cider$data

# Cross-validated AUC (mean ± SD, 10-fold x 10-repeat CV, computed in
# 9_roc.ipynb) -- this, not the in-sample AUC (cell below), is what Figure 7
# Panel C's title shows, so it reads identically to Figure 5B's "TMB + CIDER
# Score" label ("AUC x.xx ± y.yy").
cv_auc_label <- paste0(tmb_cider$cv_auc_mean, " ± ", tmb_cider$cv_auc_sd)

message("Training samples (from 9_roc.ipynb): ", nrow(train_df),
        " (", sum(train_df$response), " responders, ",
        sum(train_df$response == 0), " non-responders)")

summary(model)

# 2 - Model evaluation

AUC on the training set (in-sample) -- a diagnostic only. The AUC actually displayed in this notebook’s Figure 7 Panel C title is the cross-validated cv_auc_label (set above), matching Figure 5B’s "TMB + CIDER Score" label format exactly.

In [ ]:
train_pred <- predict(model, newdata = train_df, type = "response")

# In-sample AUC on the training set -- a diagnostic value only (not shown in
# the figure; see cv_auc_label above for the number Panel C's title uses).
# Reused from 9_roc.ipynb (tmb_cider$auc_in_sample) rather than recomputed
# here, since same model + same data gives the same number either way.
auc_val <- tmb_cider$auc_in_sample
message("Training AUC (in-sample, diagnostic only): ", auc_val)
message("Figure 7 Panel C / Figure 5B AUC (CV mean ± SD): ", cv_auc_label)

# Coefficient table
coef_df <- as.data.frame(summary(model)$coefficients) |>
  tibble::rownames_to_column("term") |>
  dplyr::rename(estimate = Estimate, std_error = `Std. Error`,
                z_value = `z value`, p_value = `Pr(>|z|)`) |>
  mutate(
    OR      = round(exp(estimate), 3),
    p_value = round(p_value, 4)
  )

print(coef_df)

# 3 - Predicted probability by actual response status

Violin plot of predicted probabilities in the CPI training cohort,
split by actual response (Responder = PR/CR, Non-Responder = SD/PD/NE).
Validates that the model discriminates the two groups before applying
it to the full Hartwig database.

In [ ]:
# Add in-sample predictions to training data
train_df$pred_prob <- train_pred

p_response_prob <- train_df |>
  mutate(
    response_label = factor(
      ifelse(response == 1, "Responder", "Non-Responder"),
      levels = c("Responder", "Non-Responder")
    )
  ) |>
  ggplot(aes(x = response_label, y = pred_prob, fill = response_label)) +
  geom_violin(alpha = 0.7, trim = TRUE) +
  geom_boxplot(width = 0.15, outlier.shape = NA, alpha = 0.7) +
  geom_hline(yintercept = 0.5, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.5) +
  # Shared project-wide Responder/Non-Responder palette (helpers/themes.r)
  # rather than hardcoded colors[5]/[6] -- keeps this panel's colours
  # consistent with every other Responder-coloured figure in the paper.
  scale_fill_manual(
    values = responder_colors,
    guide  = "none"
  ) +
  scale_y_continuous(
    limits = c(0, 1),
    labels = scales::percent_format(accuracy = 1)
  ) +
  go_theme +
  theme(plot.subtitle = element_text(hjust = 0.5)) +
  labs(
    title    = "Predicted Response Probability by Actual Response",
    subtitle = paste0("ICI-treated patients, n = ", format(nrow(train_df), big.mark = ",")),
    x        = NULL,
    y        = "Predicted Response Probability"
  ) +
  tags_main

set_plot_dims(4, 5); p_response_prob

# 4 - Predict on full Hartwig database

Apply model to all patients in `ready.rds` (T-cell group only).
Patients missing either predictor receive NA and are excluded from the landscape.

In [ ]:
# Prediction dataset: one row per patient (T-cell group only, deduplicated).
# Same predictor transforms as the shared model (9_roc.ipynb): cider_log2 =
# log2(CIDER + 1), tmb_log2 = log2(TMB + 1) -- tmbPerMb is raw here (not
# pre-transformed upstream, see 0_prepare/3_meta.ipynb).
predict_df <- ready |>
  filter(gp == "T-cell", cohort_pan %in% PAN_LEVELS) |>
  distinct(sampleId, .keep_all = TRUE) |>
  mutate(
    cider_log2 = log2(cider_score_depth_adj + 1),
    tmb_log2   = log2(tmbPerMb + 1)
  ) |>
  select(sampleId, cohort_pan, cider_log2, tmb_log2, cpi) |>
  drop_na(cider_log2, tmb_log2)

predict_df$pred_prob <- predict(model, newdata = predict_df, type = "response")

message("Patients with predictions: ", nrow(predict_df))
message("Patients excluded (missing predictor): ",
        sum(ready$gp == "T-cell" & ready$cohort_pan %in% PAN_LEVELS) - nrow(predict_df))

# 5 - Predicted response probability landscape

Boxplot of predicted ICI response probabilities across all Hartwig cancer types.
Cohorts ordered by median predicted probability (descending).
ICI-treated cohort is highlighted to show where the training population sits.

In [ ]:
# ── Panel B: CPI cohorts — predicted probability by response status ────────────
panel_a_df <- cpi |>
  dplyr::filter(gp == "T-cell", cohort_broad %in% BROAD_LEVELS_OTHER_SPLIT) |>
  dplyr::transmute(
    sampleId,
    cohort_broad,
    cohort_broad_go,
    response_int = as.integer(response),
    cider_log2   = log2(cider_score_depth_adj + 1),
    tmb_log2     = log2(tmbPerMb + 1)
  ) |>
  tidyr::drop_na(response_int, cider_log2, tmb_log2)

panel_a_df$pred_prob <- predict(model, newdata = panel_a_df, type = "response")

panel_a_df <- panel_a_df |>
  dplyr::mutate(
    response_label = factor(
      ifelse(response_int == 1, "Responder", "Non-Responder"),
      levels = c("Non-Responder", "Responder")
    ),
    cohort_broad_go = factor(cohort_broad_go, levels = levels(cpi$cohort_broad_go))
  )

n_panel_a <- panel_a_df |>
  dplyr::filter(cohort_broad == "ICI-Treated") |>
  dplyr::distinct(sampleId) |>
  nrow()

# Per-cohort median labels — black, above the median bar (matching Panel C style)
med_labels_a <- panel_a_df |>
  dplyr::group_by(cohort_broad_go, response_label) |>
  dplyr::summarise(med = median(pred_prob, na.rm = TRUE), .groups = "drop") |>
  dplyr::mutate(med_label = sprintf("%.0f%%", med * 100))

# Per-cohort n labels (responder / non-responder counts at bottom)
count_labels_a <- panel_a_df |>
  dplyr::group_by(cohort_broad_go, response_label) |>
  dplyr::summarise(n = dplyr::n(), .groups = "drop") |>
  dplyr::mutate(n_label = paste0("n = ", n))

# Per-cohort Wilcoxon p-values (actual values, not truncated)
pvals_a <- panel_a_df |>
  dplyr::group_by(cohort_broad_go) |>
  dplyr::summarise(
    p     = wilcox.test(pred_prob ~ response_label)$p.value,
    y_pos = max(pred_prob, na.rm = TRUE) * 1.05,
    .groups = "drop"
  ) |>
  dplyr::mutate(
    p_label = dplyr::case_when(
      p < 0.001 ~ sprintf("p = %.2e", p),
      p < 0.05  ~ sprintf("p = %.3f", p),
      TRUE      ~ sprintf("p = %.2f",  p)
    )
  )

p_panel_a <- panel_a_df |>
  ggplot2::ggplot(ggplot2::aes(x = cohort_broad_go, y = pred_prob, fill = response_label)) +
  # Near-zero-probability band shaded in the "negative/Non-Responder" grey
  # (responder_colors), not hardcoded colors[5] -- same semantic as the
  # Responder/Non-Responder fill below.
  ggplot2::annotate("rect",
           xmin = -Inf, xmax = Inf, ymin = 0, ymax = 0.05,
           fill = unname(responder_colors["Non-Responder"]), alpha = 0.08) +
  ggplot2::geom_hline(yintercept = 0.5, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.5) +
  ggplot2::geom_jitter(
    data     = panel_a_df |>
      dplyr::group_by(cohort_broad_go, response_label) |>
      dplyr::slice_sample(n = 150) |>
      dplyr::ungroup(),
    ggplot2::aes(colour = response_label),
    position = ggplot2::position_jitterdodge(jitter.width = 0.12, dodge.width = 0.8),
    alpha = 0.2, size = 0.4, show.legend = FALSE
  ) +
  ggplot2::scale_colour_manual(
    values = responder_colors, guide = "none"
  ) +
  ggplot2::geom_boxplot(outlier.shape = NA, alpha = 0.7,
               position = ggplot2::position_dodge(width = 0.8)) +
  # Median % labels — black, above the median bar (Panel C style)
  ggplot2::geom_text(
    data        = med_labels_a,
    ggplot2::aes(x = cohort_broad_go, y = med, label = med_label, group = response_label),
    inherit.aes = FALSE,
    position    = ggplot2::position_dodge(width = 0.8),
    vjust = -0.5, size = 2.2, fontface = "bold", colour = COL_INK
  ) +
  # n = X counts, sitting just above the x-axis (vjust = 1.3 previously pushed
  # these BELOW y = 0.01, off the visible plot area, since this panel's
  # y-scale has zero bottom expansion -- vjust = 0 grows the text upward from
  # its anchor instead, same convention as Panel C's tier-count labels below)
  ggplot2::geom_text(
    data        = count_labels_a,
    ggplot2::aes(x = cohort_broad_go, y = 0.015, label = n_label, group = response_label),
    inherit.aes = FALSE,
    position    = ggplot2::position_dodge(width = 0.8),
    vjust = 0, size = 2.1, colour = COL_INK
  ) +
  # Actual p-values above the boxes
  ggplot2::geom_text(
    data        = pvals_a,
    ggplot2::aes(x = cohort_broad_go, y = y_pos, label = p_label),
    inherit.aes = FALSE,
    vjust = 0, size = 2.8, colour = COL_INK
  ) +
  ggplot2::scale_fill_manual(
    values = responder_colors,
    name   = NULL
  ) +
  ggplot2::scale_y_continuous(
    limits = c(0, NA),
    breaks = c(0.05, seq(0, 1, by = 0.2)),
    labels = scales::percent_format(accuracy = 1),
    expand = ggplot2::expansion(mult = c(0, 0.12))
  ) +
  ggplot2::scale_x_discrete(guide = ggplot2::guide_axis(n.dodge = 2)) +
  go_theme +
  ggplot2::theme(
    plot.subtitle   = ggplot2::element_text(hjust = 0.5),
    legend.position = "right"
  ) +
  ggplot2::guides(fill = ggplot2::guide_legend(direction = "vertical", title = NULL)) +
  ggplot2::labs(
    title    = "Predicted Response Probability — ICI-Treated Cohorts",
    subtitle = paste0("CPI patients with known response, n = ",
                      format(n_panel_a, big.mark = ",")),
    x        = NULL,
    y        = "Predicted Response Probability"
  ) +
  tags_supp

set_plot_dims(8, 5); p_panel_a
saveRDS(p_panel_a, paste0(CIDER_PROJ_DIR, "figs/predicted_response_cpi_cohorts.rds"))


# 6 - Predicted response landscape — untreated patients

Apply the fitted model to all untreated Hartwig patients. Cohorts are
ordered by median predicted probability. The shaded red region (0–5%)
marks cancer types with very low predicted ICI responsiveness.

In [ ]:
# ── Panel C: Untreated Hartwig patients — predicted response landscape ─────────
panel_b_df <- predict_df |>
  filter(!cpi, cohort_pan %in% PAN_LEVELS) |>
  select(sampleId, cohort_pan, pred_prob)

cohort_summary_b <- panel_b_df |>
  group_by(cohort_pan) |>
  summarise(
    n   = n(),
    med = median(pred_prob, na.rm = TRUE),
    .groups = "drop"
  ) |>
  arrange(desc(med)) |>
  mutate(label = paste0(cohort_pan, "\n(N = ", format(n, big.mark = ",", trim = TRUE), ")"))

panel_b_df <- panel_b_df |>
  left_join(select(cohort_summary_b, cohort_pan, label), by = "cohort_pan") |>
  mutate(label = factor(label, levels = cohort_summary_b$label))

med_labels_b <- cohort_summary_b |>
  mutate(
    med_label = sprintf("%.0f%%", med * 100),
    label     = factor(label, levels = cohort_summary_b$label)
  )

p_panel_b <- panel_b_df |>
  ggplot(aes(x = label, y = pred_prob)) +
  # Reference bands use the same Responder (orange) / Non-Responder (grey)
  # palette as every other response-coloured panel, rather than hardcoded
  # colors[5]/[6].
  annotate("rect",
           xmin = -Inf, xmax = Inf, ymin = 0, ymax = 0.05,
           fill = unname(responder_colors["Non-Responder"]), alpha = 0.08) +
  annotate("rect",
           xmin = -Inf, xmax = Inf, ymin = 0.5, ymax = 1,
           fill = unname(responder_colors["Responder"]), alpha = 0.05) +
  geom_hline(yintercept = 0.5, linetype = "dashed", colour = COL_NEGATIVE, linewidth = 0.5) +
  geom_jitter(
    data  = panel_b_df |>
      dplyr::group_by(label) |>
      dplyr::slice_sample(n = 150) |>
      dplyr::ungroup(),
    width = 0.18, alpha = 0.12, size = 0.35, colour = COL_NEGATIVE
  ) +
  # Emphasise predicted non-responders (<5%) and likely responders (>50%)
  geom_jitter(
    data  = panel_b_df |>
      dplyr::filter(pred_prob < 0.05 | pred_prob > 0.50),
    aes(colour = dplyr::if_else(pred_prob < 0.05, "low", "high")),
    width = 0.18, alpha = 0.45, size = 0.8
  ) +
  scale_colour_manual(
    values = c("low" = unname(responder_colors["Non-Responder"]),
              "high" = unname(responder_colors["Responder"])),
    guide  = "none"
  ) +
  geom_boxplot(fill = COL_NEGATIVE, colour = COL_INK, outlier.shape = NA, alpha = 0.7) +
  geom_text(
    data        = med_labels_b,
    aes(x = label, y = med, label = med_label),
    inherit.aes = FALSE,
    vjust = -0.5, size = 2.2, fontface = "bold", colour = COL_INK
  ) +
  scale_y_continuous(
    limits = c(0, 1),
    breaks = c(0, 0.05, 0.5, 1),
    labels = scales::percent_format(accuracy = 1),
    expand = expansion(mult = c(0.01, 0.03))
  ) +
  go_theme +
  theme(
    axis.text.x   = element_text(angle = 0, hjust = 0.5, size = 9),
    plot.subtitle = element_text(hjust = 0.5)
  ) +
  labs(
    # "Extrapolated" (not "Predicted") -- the model was fit and cross-validated
    # entirely on ICI-treated patients (see 9_roc.ipynb); applying it here to
    # patients who never received ICI is an out-of-distribution projection,
    # not a validated prediction for this population. The AUC in parentheses
    # is the model's own CV performance on its ICI-treated training/validation
    # population (Figure 5B's "TMB + CIDER Score" label) -- it says nothing
    # about accuracy in the untreated patients this panel actually plots,
    # which is exactly the caveat "with ICI Treated Patients" calls out.
    title    = paste0("Extrapolated ICI Response Probability | Model AUC (",
                      cv_auc_label, ") within ICI Treated Patients"),
    subtitle = paste0("All Hartwig patients without ICI treatment, n = ",
                      format(nrow(panel_b_df), big.mark = ",")),
    x        = "Cohort",
    y        = "Extrapolated Response Probability"
  ) +
  tags_supp


# ── Emphasize predicted Low/High response counts directly on the boxplot ──────
# Simpler alternative to a separate stacked tier-composition panel: annotate
# each cohort's box with the raw patient count in the two extreme tiers only
# (Low <5%, High >50%), coloured to match the shaded reference bands/jitter
# above (same responder_colors Non-Responder/Responder pair). The
# unremarkable Mid tier isn't called out.
tier_counts_b <- panel_b_df |>
  dplyr::group_by(label) |>
  dplyr::summarise(
    n_low  = sum(pred_prob < 0.05, na.rm = TRUE),
    n_high = sum(pred_prob > 0.50, na.rm = TRUE),
    .groups = "drop"
  )

p_panel_b <- p_panel_b +
  # Skip the label entirely where the count is 0 -- an empty tier isn't
  # worth calling out, and dropping it thins out an otherwise busy panel.
  # Colours match the shaded reference bands/jitter above via the shared
  # responder_colors palette (Non-Responder grey / Responder orange) rather
  # than hardcoded colors[5]/[6].
  geom_text(
    data        = dplyr::filter(tier_counts_b, n_low > 0),
    aes(x = label, y = 0.006, label = paste0("n = ", n_low)),
    inherit.aes = FALSE, colour = unname(responder_colors["Non-Responder"]),
    size = 2.6, fontface = "bold", vjust = 0
  ) +
  geom_text(
    data        = dplyr::filter(tier_counts_b, n_high > 0),
    aes(x = label, y = 0.988, label = paste0("n = ", n_high)),
    inherit.aes = FALSE, colour = unname(responder_colors["Responder"]),
    size = 2.6, fontface = "bold", vjust = 1
  )

set_plot_dims(14, 6); p_panel_b
saveRDS(p_panel_b, paste0(CIDER_PROJ_DIR, "figs/predicted_response_landscape_no_overlay.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/predicted_response_landscape_no_overlay.png"),
       p_panel_b, width = 14, height = 6, dpi = 300)
message("Saved predicted_response_landscape_no_overlay.png/.rds")

# ── External validation ORR overlay (literature response rates) ───────────────
# Each cohort that has a published ORR gets a subtle red tick at that y-level.
orr_overlay <- tibble::tribble(
  ~cohort_pan,  ~orr_pct,
  "CRC-MSI",    38.8,
  "SKCM",       37.0,
  "BLCA",       23.0,
  "STAD",       20.4,
  "KIRC",       19.9,
  "ENDO",       18.6,
  "NSCLC",      17.1,
  "CESC",       15.3,
  "LIHC",       15.2,
  "ESOG",       15.0,
  "MESO",       13.5,
  "OV",         10.9,
  "CNS",          8.7,
  "SARC",         7.8,
  "CHOL",         7.3,
  "BRCA",         7.1,
  "PRAD",         5.0,
  "CRC-MSS",      0.0,
  "PAAD",         0.0
) |>
  dplyr::left_join(
    dplyr::select(cohort_summary_b, cohort_pan, label),
    by = "cohort_pan"
  ) |>
  dplyr::filter(!is.na(label)) |>
  dplyr::mutate(
    label    = factor(label, levels = cohort_summary_b$label),
    orr_prob = orr_pct / 100
  )

# ── Version 2: landscape WITH the external-validation overlay ─────────────────
# Overlay is intentionally light (alpha = 0.4) so it reads as a reference
# rather than competing with the boxplots, with a top-right label explaining it.
p_panel_b_overlay <- p_panel_b +
  ggplot2::geom_errorbar(
    data        = orr_overlay,
    ggplot2::aes(x = label, ymin = orr_prob, ymax = orr_prob),
    width       = 0.55,
    colour      = COL_OVERLAY,
    linewidth   = 0.9,
    alpha       = 0.4,
    inherit.aes = FALSE
  ) +
  ggplot2::geom_text(
    # 0% cohorts (e.g. no published responders) still get the red reference
    # line, but the "0%" text is dropped -- the line at the axis floor
    # already says it plainly, so the label is just clutter.
    data        = dplyr::filter(orr_overlay, orr_pct > 0),
    ggplot2::aes(x = label, y = orr_prob,
                 label = sprintf("%.0f%%", orr_pct)),
    colour      = COL_OVERLAY,
    size        = 2.1,
    vjust       = -0.5,
    fontface    = "bold",
    inherit.aes = FALSE
  ) +
  ggplot2::annotate(
    "text", x = Inf, y = Inf,
    label    = "Red: published literature ORR (external validation)",
    colour   = COL_OVERLAY,
    hjust    = 1.02,
    vjust    = 1.6,
    size     = 2.6,
    fontface = "italic"
  )

set_plot_dims(14, 6); p_panel_b_overlay
saveRDS(p_panel_b_overlay, paste0(CIDER_PROJ_DIR, "figs/predicted_response_landscape.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/predicted_response_landscape.png"),
       p_panel_b_overlay, width = 14, height = 6, dpi = 300)
message("Saved predicted_response_landscape.png/.rds (with overlay — feeds Figure 7 assembly)")

In [ ]:
# ── Literature ORR vs Model Predicted Response ────────────────────────────────
# Compare median predicted response probability (Panel C) against published
# objective response rates (ORR) from Jardim et al. / systematic review.
# Mapping: Hartwig cohort_pan labels → nearest TCGA tumour type ORR.
# NSCLC = mean(LUAD 16.7%, LUSC 17.4%); ESOG = ESCA; KIRC = KICH/KIRC/KIRP;
# ENDO = UCEC; CNS = GBM. Cohorts without a clear match (CUP, Other, LNET) excluded.

orr_lit <- tibble::tribble(
  ~cohort_pan,  ~orr_pct,  ~label,
  "CRC-MSI",    38.8,      "CRC-MSI",
  "SKCM",       37.0,      "SKCM",
  "BLCA",       23.0,      "BLCA",
  "STAD",       20.4,      "STAD",
  "KIRC",       19.9,      "KIRC",
  "ENDO",       18.6,      "ENDO",
  "NSCLC",      17.1,      "NSCLC",
  "CESC",       15.3,      "CESC",
  "LIHC",       15.2,      "LIHC",
  "ESOG",       15.0,      "ESOG",
  "MESO",       13.5,      "MESO",
  "OV",         10.9,      "OV",
  "CNS",         8.7,      "CNS",
  "SARC",        7.8,      "SARC",
  "CHOL",        7.3,      "CHOL",
  "BRCA",        7.1,      "BRCA",
  "PRAD",        5.0,      "PRAD",
  "CRC-MSS",     0.0,      "CRC-MSS",
  "PAAD",        0.0,      "PAAD"
)

# Median predicted probability per cohort (from Panel C data)
pred_med <- panel_b_df |>
  dplyr::group_by(cohort_pan) |>
  dplyr::summarise(pred_med_pct = median(pred_prob, na.rm = TRUE) * 100, .groups = "drop")

# Join
orr_df <- dplyr::inner_join(orr_lit, pred_med, by = "cohort_pan")

# Spearman correlation
corr_val <- cor(orr_df$pred_med_pct, orr_df$orr_pct, method = "spearman")
corr_lbl <- sprintf("r[s] == %.2f", corr_val)

p_orr <- ggplot2::ggplot(orr_df, ggplot2::aes(x = pred_med_pct, y = orr_pct, label = label)) +
  ggplot2::geom_abline(intercept = 0, slope = 1, linetype = "solid", colour = COL_NEGATIVE, linewidth = 0.4) +
  ggplot2::geom_smooth(method = "lm", se = TRUE, colour = COL_NEGATIVE, fill = COL_NEGATIVE, linewidth = 0.6) +
  # Plain single-series scatter accent (no categorical/overlay meaning) --
  # COL_ACCENT rather than COL_OVERLAY, which is reserved for the red
  # literature-ORR reference ticks in the panel above (geom_errorbar/geom_text
  # with colour = COL_OVERLAY).
  ggplot2::geom_point(size = 2.5, colour = COL_ACCENT, alpha = 0.8) +
  ggrepel::geom_text_repel(size = 2.8, colour = COL_INK, max.overlaps = 20,
                            box.padding = 0.3) +
  ggplot2::annotate("text", x = Inf, y = -Inf,
                    label = corr_lbl, parse = TRUE,
                    hjust = 1.1, vjust = -0.8, size = 3.5, colour = COL_INK) +
  ggplot2::scale_x_continuous(labels = scales::percent_format(scale = 1),
                               limits = c(0, NA),
                               expand = ggplot2::expansion(mult = c(0, 0.05))) +
  ggplot2::scale_y_continuous(labels = scales::percent_format(scale = 1),
                               expand = ggplot2::expansion(mult = c(0.05, 0.1))) +
  go_theme +
  ggplot2::labs(
    title    = "Model Predicted vs Literature ORR by Cancer Type",
    subtitle = "Spearman correlation — Hartwig CIDER+TMB model vs published ICI response rates",
    x        = "Median Predicted Response Probability",
    y        = "Objective Response Rate (Literature, %)"
  )

set_plot_dims(6, 5); p_orr
saveRDS(p_orr, paste0(CIDER_PROJ_DIR, "figs/predicted_vs_orr.rds"))
ggsave(paste0(CIDER_PROJ_DIR, "figs/predicted_vs_orr.png"),
       p_orr, width = 6, height = 5, dpi = 300)
message("Saved predicted_vs_orr.png")